# 函数调用与工具使用

> 前面几节讲的是怎么训练一个模型：训练损失怎么算、数据怎么处理、参数怎么高效更新。训完的模型已经能流畅生成文本、回答常识问题。
>
> 但有一类问题它答不了——今天的天气、357 乘以 289、当前某个仓库的提交数。这些都需要外部信息或外部计算。这一节讲 Function Call：通过专门的数据构造和训练，让模型学会输出结构化的调用请求，再由外部程序执行后把结果回填给模型。

LLM 的训练数据有一个时间截止点，之后发生的事模型不知道。LLM 也不擅长精确的数值计算——让它直接算 357 乘以 289，结果可能看起来合理但其实是错的。这两个限制叠加起来，意味着纯文本生成在很多实际场景里不够用。

Function Call 的做法是给模型加一个外部接口。模型本身不执行任何操作，它只生成一段结构化的 JSON，里面写明要调用什么函数、传什么参数。这段 JSON 由外部程序解析、执行（比如真的去查天气 API、调用计算器），执行结果再被拼回对话历史，模型基于结果生成最终的自然语言回复。

这种能力是通过专门训练获得的，模型不是天生就会输出 JSON。下面从问题出发，看 Function Call 的完整流程，再看怎么训练这种能力。

## 本节要点

通过这一节的学习，以下问题应该能够回答：

1. LLM 在什么场景下需要外部工具？
2. Function Call 的完整流程是什么？
3. 一次对话里需要多次工具调用时，怎么处理并行和串行？
4. 怎么训练一个支持 Function Call 的模型？
5. Function Call 的常见失败模式有哪些？
6. 从 Function Call 到 MCP，工具调用生态的演化路径是什么？

## 1. 为什么需要 Function Call

LLM 的知识来自训练数据，训练数据有一个时间截止点。问模型「今天北京多少度」，它会基于训练时见过的天气统计给出一个笼统的回答，但不可能知道今天的真实天气。这是第一个限制——知识截止。

第二个限制是数值计算。Transformer 的基本运算是矩阵乘法和 softmax，并不擅长精确的多位数乘法。让模型直接算 357 × 289，结果常常是错的。模型给出的答案看起来合理（数量级对、个位数对），但中间几位数字往往出错。

下面用代码模拟这两种失败。

In [1]:
# 模拟 LLM 的两个典型失败：知识截止 + 数值计算错误

# 场景 1：知识截止
# 模型的训练数据是过去的，无法知道"今天"的真实数据
def fake_llm_today_weather():
    # 一个不懂工具的 LLM 被问到今天的天气
    return "北京今天的天气我无法实时查询，但通常这个季节气温在 20-30°C 之间。"

print("=== 失败 1：知识截止 ===")
print("用户：北京今天多少度？")
print(f"模型：{fake_llm_today_weather()}")
print("问题：模型不知道今天的真实天气，只能给一个笼统的猜测")
print()

# 场景 2：数值计算错误
# 模拟一个会算错的"模型"——它输出一个看起来合理但错的答案
def fake_llm_multiply(a, b):
    # 模拟 LLM 算乘法，可能算错
    correct = a * b
    wrong = correct + 1234  # 故意给一个错的答案
    return wrong

a, b = 357, 289
correct = a * b
llm_answer = fake_llm_multiply(a, b)
print("=== 失败 2：数值计算错误 ===")
print(f"用户：{a} × {b} 等于多少？")
print(f"模型：{llm_answer}")
print(f"正确：{correct}")
print(f"差距：模型答案比正确答案多了 {llm_answer - correct}")
print()
print("结论：这两类问题靠模型自己是解决不了的。需要给它一个外部接口。")

=== 失败 1：知识截止 ===
用户：北京今天多少度？
模型：北京今天的天气我无法实时查询，但通常这个季节气温在 20-30°C 之间。
问题：模型不知道今天的真实天气，只能给一个笼统的猜测

=== 失败 2：数值计算错误 ===
用户：357 × 289 等于多少？
模型：104407
正确：103173
差距：模型答案比正确答案多了 1234

结论：这两类问题靠模型自己是解决不了的。需要给它一个外部接口。


## 2. Function Call 的完整流程

Function Call 的核心思路是把"执行"和"决策"分开：

- 决策（该不该调用工具、调用哪个、传什么参数）由模型完成
- 执行（真的去查天气、算数、查数据库）由外部程序完成

完整流程分六步。下面用一个具体例子走一遍：用户问「北京今天多少度？」。

第一步：定义工具。每个工具是一个 JSON 描述，告诉模型这个工具叫什么、做什么、需要什么参数。
第二步：把工具描述注入到 system prompt。模型在生成回复前会看到这段描述。
第三步：模型生成回复。如果它判断需要调用工具，输出的不是自然语言，而是一段结构化 JSON。
第四步：外部程序解析这段 JSON，调用对应函数。
第五步：把执行结果包装成一个新的对话消息，加进对话历史。
第六步：模型基于完整对话历史（含工具结果）生成最终的自然语言回复。

In [2]:
# 第一步：定义可用工具
# 每个工具是一个 dict，包含 name/description/parameters 三部分

import json

tools = [
    {
        "name": "get_weather",
        "description": "查询指定城市的当前天气",
        "parameters": {
            "city": {"type": "string", "description": "城市名"},
            "unit": {"type": "string", "enum": ["celsius", "fahrenheit"], "description": "温度单位"}
        }
    },
    {
        "name": "calculate",
        "description": "执行数学表达式计算",
        "parameters": {
            "expression": {"type": "string", "description": "数学表达式"}
        }
    }
]

print("=== 已定义的工具 ===")
for t in tools:
    print(f"\n工具名：{t['name']}")
    print(f"  说明：{t['description']}")
    print(f"  参数：")
    for pname, pinfo in t['parameters'].items():
        print(f"    {pname} ({pinfo['type']}): {pinfo['description']}")

=== 已定义的工具 ===

工具名：get_weather
  说明：查询指定城市的当前天气
  参数：
    city (string): 城市名
    unit (string): 温度单位

工具名：calculate
  说明：执行数学表达式计算
  参数：
    expression (string): 数学表达式


In [3]:
# 第二步：把工具描述拼进 system prompt
# 这一段是模型可见的，模型据此判断要不要调用工具、调用哪个

def build_system_prompt(tools):
    # 根据工具列表生成系统提示
    lines = ["你是一个助手，可以使用以下工具。当需要外部信息时，请输出 JSON 格式的调用请求。", ""]
    lines.append("可用工具：")
    for t in tools:
        lines.append(f"\n工具：{t['name']}")
        lines.append(f"  说明：{t['description']}")
        lines.append(f"  参数：")
        for pname, pinfo in t['parameters'].items():
            lines.append(f"    - {pname}: {pinfo['description']}")
    return "\n".join(lines)

system_prompt = build_system_prompt(tools)
print("=== System Prompt（模型会看到这段）===")
print(system_prompt)

=== System Prompt（模型会看到这段）===
你是一个助手，可以使用以下工具。当需要外部信息时，请输出 JSON 格式的调用请求。

可用工具：

工具：get_weather
  说明：查询指定城市的当前天气
  参数：
    - city: 城市名
    - unit: 温度单位

工具：calculate
  说明：执行数学表达式计算
  参数：
    - expression: 数学表达式


In [4]:
# 第三步：模拟模型生成 JSON 调用请求
# 实际中这是模型 forward 出来的 token 序列；这里手动模拟一个合理的输出

user_query = "北京今天多少度？"

# 假设模型经过训练后，面对这个用户问题和上面的工具列表，会输出这样的 JSON
model_output = json.dumps({
    "tool_call": {
        "name": "get_weather",
        "arguments": {"city": "北京", "unit": "celsius"}
    }
}, ensure_ascii=False, indent=2)

print("=== 模型输出 ===")
print(f"用户：{user_query}")
print("模型没有输出自然语言，而是输出 JSON：")
print(model_output)
print()
print("关键观察：模型判断需要外部信息，于是选择 get_weather 工具，参数填了 city='北京'。")

=== 模型输出 ===
用户：北京今天多少度？
模型没有输出自然语言，而是输出 JSON：
{
  "tool_call": {
    "name": "get_weather",
    "arguments": {
      "city": "北京",
      "unit": "celsius"
    }
  }
}

关键观察：模型判断需要外部信息，于是选择 get_weather 工具，参数填了 city='北京'。


In [5]:
# 第四步 + 第五步：外部程序执行工具，结果回填进对话历史

# 模拟工具的真实实现
def execute_tool(name, arguments):
    # 根据工具名执行对应的函数
    if name == "get_weather":
        # 实际中这里会调用真实的天气 API
        return {"temperature": 22, "condition": "晴", "humidity": 45}
    elif name == "calculate":
        # 实际中这里会调用真实的计算器
        return {"result": eval(arguments["expression"])}
    return {"error": f"未知工具：{name}"}

# 解析模型输出的 JSON，执行对应工具
parsed = json.loads(model_output)
call = parsed["tool_call"]
tool_result = execute_tool(call["name"], call["arguments"])

print("=== 工具执行 ===")
print(f"调用：{call['name']}({call['arguments']})")
print(f"返回：{tool_result}")
print()

# 构造完整的对话历史，把工具结果作为新的消息加进去
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_query},
    {"role": "assistant", "content": model_output},
    {"role": "tool", "name": call["name"], "content": json.dumps(tool_result, ensure_ascii=False)},
]

print("=== 完整对话历史（此时已含工具结果）===")
for m in messages:
    role = m["role"]
    content = m["content"]
    if len(content) > 80:
        content = content[:80] + "..."
    print(f"[{role}] {content}")

=== 工具执行 ===
调用：get_weather({'city': '北京', 'unit': 'celsius'})
返回：{'temperature': 22, 'condition': '晴', 'humidity': 45}

=== 完整对话历史（此时已含工具结果）===
[system] 你是一个助手，可以使用以下工具。当需要外部信息时，请输出 JSON 格式的调用请求。

可用工具：

工具：get_weather
  说明：查询指定城市的当前...
[user] 北京今天多少度？
[assistant] {
  "tool_call": {
    "name": "get_weather",
    "arguments": {
      "city": "...
[tool] {"temperature": 22, "condition": "晴", "humidity": 45}


In [6]:
# 第六步：模型基于完整对话历史生成最终自然语言回复

# 模拟模型生成的最终回复
final_reply = "北京今天 22°C，晴天。"

print("=== 最终回复 ===")
print(f"模型：{final_reply}")
print()
print("关键观察：模型看到工具返回的温度=22、天气=晴后，把它们组合成一句自然语言。")
print("用户感知到的是一句普通的回答，工具调用的细节被隐藏了。")

=== 最终回复 ===
模型：北京今天 22°C，晴天。

关键观察：模型看到工具返回的温度=22、天气=晴后，把它们组合成一句自然语言。
用户感知到的是一句普通的回答，工具调用的细节被隐藏了。


## 3. 多工具场景

实际场景里，一个问题经常需要多个工具。比如用户问「北京今天多少度？另外 357 × 289 等于多少？」，模型需要同时调用 get_weather 和 calculate 两个工具。

这里有两种情况，处理方式不同。

并行调用：两个工具之间没有依赖关系，谁先调谁后调都不影响结果。模型可以在一次输出里同时生成多个调用请求，外部程序并行执行后，把所有结果一次性回填。这种方式延迟低。

串行调用：第二个工具的参数依赖第一个工具的结果。模型必须先调第一个工具，看到结果后再决定调不调第二个、传什么参数。比如「查北京今天多少度，然后告诉我该穿什么」——第二个调用（推荐穿着）依赖第一个调用（天气）的结果。这种情况需要分多轮进行。

In [7]:
# 并行调用：一次生成多个工具调用请求

user_query = "北京今天多少度？另外 357 × 289 等于多少？"

# 模型一次性输出多个调用
model_output = json.dumps({
    "tool_calls": [
        {"name": "get_weather", "arguments": {"city": "北京", "unit": "celsius"}},
        {"name": "calculate", "arguments": {"expression": "357*289"}}
    ]
}, ensure_ascii=False, indent=2)

print("=== 并行调用 ===")
print(f"用户：{user_query}")
print("模型输出：")
print(model_output)
print()

# 外部程序解析并依次执行（实际中可以并行）
parsed = json.loads(model_output)
for call in parsed["tool_calls"]:
    result = execute_tool(call["name"], call["arguments"])
    print(f"执行 {call['name']} → {result}")

print()
print("关键观察：模型一次输出了两个调用，两个调用之间没有依赖。")
print("外部程序可以并行执行，把两个结果同时回填给模型。")

=== 并行调用 ===
用户：北京今天多少度？另外 357 × 289 等于多少？
模型输出：
{
  "tool_calls": [
    {
      "name": "get_weather",
      "arguments": {
        "city": "北京",
        "unit": "celsius"
      }
    },
    {
      "name": "calculate",
      "arguments": {
        "expression": "357*289"
      }
    }
  ]
}

执行 get_weather → {'temperature': 22, 'condition': '晴', 'humidity': 45}
执行 calculate → {'result': 103173}

关键观察：模型一次输出了两个调用，两个调用之间没有依赖。
外部程序可以并行执行，把两个结果同时回填给模型。


In [8]:
# 串行调用：第二个调用依赖第一个的结果

user_query = "北京今天多少度？穿什么合适？"

# 第一轮：模型先调用天气工具
first_call = json.dumps({
    "tool_call": {"name": "get_weather", "arguments": {"city": "北京", "unit": "celsius"}}
}, ensure_ascii=False)

print("=== 串行调用 ===")
print(f"用户：{user_query}")
print("\n第 1 轮模型输出：")
print(first_call)

weather_result = execute_tool("get_weather", {"city": "北京", "unit": "celsius"})
print(f"\n工具结果：{weather_result}")
print()

# 第二轮：模型看到天气结果后，决定调用穿衣建议工具
wardrobe_tool_result = {"suggestion": "长袖衬衫加薄外套"}
print("第 2 轮模型输出：基于天气 22°C 晴，调用穿衣建议工具")
print(f"工具结果：{wardrobe_tool_result}")
print()
print("关键观察：第二轮调用依赖第一轮的结果——模型必须先知道温度，才能给出穿衣建议。")
print("这种依赖关系让串行调用必须分多轮完成，延迟比并行高。")

=== 串行调用 ===
用户：北京今天多少度？穿什么合适？

第 1 轮模型输出：
{"tool_call": {"name": "get_weather", "arguments": {"city": "北京", "unit": "celsius"}}}

工具结果：{'temperature': 22, 'condition': '晴', 'humidity': 45}

第 2 轮模型输出：基于天气 22°C 晴，调用穿衣建议工具
工具结果：{'suggestion': '长袖衬衫加薄外套'}

关键观察：第二轮调用依赖第一轮的结果——模型必须先知道温度，才能给出穿衣建议。
这种依赖关系让串行调用必须分多轮完成，延迟比并行高。


## 4. 训练一个支持 Function Call 的模型

模型不会天生就输出 JSON 格式的工具调用。一个普通的预训练模型面对「调用 get_weather 工具」的请求，会输出一段看起来像 JSON 但格式不严格、参数也可能编造的文本。要让它可靠地输出合法的 JSON 调用，需要专门的数据构造和训练。

训练数据和普通的指令微调数据格式很像，区别在于 assistant 的回复里包含了工具调用 JSON。下面看一个具体的训练样本。

In [9]:
# 一个完整的 Function Call 训练样本

training_sample = {
    "messages": [
        {
            "role": "system",
            "content": "你是一个助手，可以使用以下工具：get_weather(city) 查询天气；calculate(expression) 计算。"
        },
        {
            "role": "user",
            "content": "北京今天多少度？"
        },
        {
            "role": "assistant",
            "content": "",
            "tool_call": {"name": "get_weather", "arguments": {"city": "北京", "unit": "celsius"}}
        },
        {
            "role": "tool",
            "name": "get_weather",
            "content": '{"temperature": 22, "condition": "晴"}'
        },
        {
            "role": "assistant",
            "content": "北京今天 22°C，晴天。"
        }
    ]
}

print("=== 一个 Function Call 训练样本 ===")
print(json.dumps(training_sample, ensure_ascii=False, indent=2))
print()
print("这个样本包含完整的对话流：system → user → assistant(工具调用) → tool(结果) → assistant(最终回复)。")
print("模型通过大量这样的样本，学会在合适的时候输出工具调用 JSON。")

=== 一个 Function Call 训练样本 ===
{
  "messages": [
    {
      "role": "system",
      "content": "你是一个助手，可以使用以下工具：get_weather(city) 查询天气；calculate(expression) 计算。"
    },
    {
      "role": "user",
      "content": "北京今天多少度？"
    },
    {
      "role": "assistant",
      "content": "",
      "tool_call": {
        "name": "get_weather",
        "arguments": {
          "city": "北京",
          "unit": "celsius"
        }
      }
    },
    {
      "role": "tool",
      "name": "get_weather",
      "content": "{\"temperature\": 22, \"condition\": \"晴\"}"
    },
    {
      "role": "assistant",
      "content": "北京今天 22°C，晴天。"
    }
  ]
}

这个样本包含完整的对话流：system → user → assistant(工具调用) → tool(结果) → assistant(最终回复)。
模型通过大量这样的样本，学会在合适的时候输出工具调用 JSON。


### 4.1 Loss 怎么算

训练 LLM 的标准做法是预测下一个 token。但在 Function Call 训练里，并不是所有 token 都该贡献 loss。system 消息、user 消息、tool 结果消息都是模型的输入，模型不应该被要求"预测"它们——它该学的是在 assistant 位置输出什么。

所以训练时会用一个 mask：把输入部分（system + user + tool 结果）对应的 token 标记为 0（不算 loss），把 assistant 输出部分（包括工具调用 JSON 和最终回复）标记为 1（算 loss）。

In [10]:
# Loss mask 演示：哪些 token 算 loss，哪些不算

# 把训练样本渲染成连续文本，并标注每个角色的 token 是否算 loss
def render_sample(sample):
    # 把多轮对话渲染成连续文本，标注每个角色是否算 loss
    parts = []
    for msg in sample["messages"]:
        role = msg["role"]
        if role == "system":
            parts.append((f"[系统] {msg['content']}\n", False))
        elif role == "user":
            parts.append((f"[用户] {msg['content']}\n", False))
        elif role == "assistant" and msg.get("tool_call"):
            tc = msg["tool_call"]
            parts.append((f"[助手-工具调用] {json.dumps(tc, ensure_ascii=False)}\n", True))
        elif role == "assistant":
            parts.append((f"[助手] {msg['content']}\n", True))
        elif role == "tool":
            parts.append((f"[工具-{msg['name']}] {msg['content']}\n", False))
    return parts

parts = render_sample(training_sample)

print("=== Loss Mask 演示 ===")
print("（True = 算 loss，False = 不算 loss）\n")
for text, mask in parts:
    label = "算 loss" if mask else "不算"
    short = text[:60].replace("\n", "") + ("..." if len(text) > 60 else "")
    print(f"[{label:6}] {short}")

print()
print("关键观察：只有 assistant 的输出（工具调用 JSON + 最终回复）算 loss。")
print("system/user/tool 都是输入，模型不需要预测它们。")

=== Loss Mask 演示 ===
（True = 算 loss，False = 不算 loss）

[不算    ] [系统] 你是一个助手，可以使用以下工具：get_weather(city) 查询天气；calculate(expres...
[不算    ] [用户] 北京今天多少度？
[算 loss] [助手-工具调用] {"name": "get_weather", "arguments": {"city": "北京"...
[不算    ] [工具-get_weather] {"temperature": 22, "condition": "晴"}
[算 loss] [助手] 北京今天 22°C，晴天。

关键观察：只有 assistant 的输出（工具调用 JSON + 最终回复）算 loss。
system/user/tool 都是输入，模型不需要预测它们。


### 4.2 训练数据从哪里来

构造训练样本是 Function Call 训练里最耗工时的部分。常见的三种数据来源：

| 来源 | 做法 | 优缺点 |
|:---|:---|:---|
| 人工标注 | 给标注员看工具列表，让他们编写"用户问题 + 期望的工具调用"对 | 质量高，但成本高、规模难放大 |
| 强模型蒸馏 | 让 GPT-4 这类已经支持工具调用的模型生成调用样本，再用作训练数据 | 规模容易放大，但受限于强模型的能力上限 |
| 真实日志 | 从已有产品的工具调用日志里筛选正确的样本 | 最贴近真实分布，但冷启动阶段没有日志 |

实际项目通常三种混用：先用人工标注做几千条高质量种子数据，再用强模型蒸馏扩到几万条，最后接入真实日志持续优化。

## 5. 错误处理与失败模式

工具调用不是一调用就成功。实际部署中会遇到各种失败，模型和外部程序都需要有能力处理。

下面列出四种最常见的失败模式，每种配一个具体例子。

In [11]:
# 失败模式 1：工具执行失败
# 比如天气 API 超时，或者用户传了一个不存在的城市

def execute_tool_with_failure(name, arguments):
    # 一个可能失败的工具执行器
    if name == "get_weather":
        city = arguments.get("city", "")
        # 模拟"城市不存在"
        if city not in ["北京", "上海", "广州"]:
            return {"error": f"未找到城市：{city}"}
        return {"temperature": 22, "condition": "晴"}
    return {"error": "未知工具"}

# 案例：模型调了一个不存在的城市
result = execute_tool_with_failure("get_weather", {"city": "亚特兰蒂斯"})
print("=== 失败模式 1：工具执行失败 ===")
print("模型调用：get_weather(city='亚特兰蒂斯')")
print(f"工具返回：{result}")
print()
print("工程对策：把错误结果原样回填给模型，让模型生成道歉或追问的回复。")
print("示例：'抱歉，未找到亚特兰蒂斯的天气信息。请问您指的是哪个城市？'")

=== 失败模式 1：工具执行失败 ===
模型调用：get_weather(city='亚特兰蒂斯')
工具返回：{'error': '未找到城市：亚特兰蒂斯'}

工程对策：把错误结果原样回填给模型，让模型生成道歉或追问的回复。
示例：'抱歉，未找到亚特兰蒂斯的天气信息。请问您指的是哪个城市？'


In [12]:
# 失败模式 2：模型选错工具
# 用户问的是计算，但模型调了天气工具

wrong_call = {"name": "get_weather", "arguments": {"city": "357*289"}}
print("=== 失败模式 2：模型选错工具 ===")
print("用户：357 × 289 等于多少？")
print(f"模型调用：{wrong_call}")
print("问题：模型把数学表达式当成了城市名")
print()
print("工程对策：在训练数据里加入更多'用户问题类型 → 正确工具'的样本。")
print("也可以在 system prompt 里强调每个工具的适用场景。")

=== 失败模式 2：模型选错工具 ===
用户：357 × 289 等于多少？
模型调用：{'name': 'get_weather', 'arguments': {'city': '357*289'}}
问题：模型把数学表达式当成了城市名

工程对策：在训练数据里加入更多'用户问题类型 → 正确工具'的样本。
也可以在 system prompt 里强调每个工具的适用场景。


In [13]:
# 失败模式 3：参数不合法
# 比如工具要求 unit 只能是 celsius 或 fahrenheit，模型传了 kelvin

bad_call = {"name": "get_weather", "arguments": {"city": "北京", "unit": "kelvin"}}
print("=== 失败模式 3：参数不合法 ===")
print(f"模型调用：{bad_call}")
print("问题：unit='kelvin' 不在 enum ['celsius', 'fahrenheit'] 内")
print()

# 工程对策：用 JSON Schema 严格校验
valid_units = ["celsius", "fahrenheit"]
actual_unit = bad_call["arguments"]["unit"]
if actual_unit not in valid_units:
    print(f"校验失败：unit 必须是 {valid_units}，但收到了 '{actual_unit}'")
    print("工程对策：在外部程序里加 schema 校验，不合法的参数直接拒绝，")
    print("并把校验错误信息回填给模型，让它重试。")

=== 失败模式 3：参数不合法 ===
模型调用：{'name': 'get_weather', 'arguments': {'city': '北京', 'unit': 'kelvin'}}
问题：unit='kelvin' 不在 enum ['celsius', 'fahrenheit'] 内

校验失败：unit 必须是 ['celsius', 'fahrenheit']，但收到了 'kelvin'
工程对策：在外部程序里加 schema 校验，不合法的参数直接拒绝，
并把校验错误信息回填给模型，让它重试。


In [14]:
# 失败模式 4：工具调用死循环
# 模型反复调用同一个工具，永远不输出最终回复

print("=== 失败模式 4：工具调用死循环 ===")
print("场景：模型连续多次调用 get_weather，每次参数都一样，从不输出最终回复")
print()

# 工程对策：设置最大调用次数
MAX_TOOL_CALLS = 3
call_count = 0
for i in range(MAX_TOOL_CALLS + 2):  # 模拟模型反复调用
    call_count += 1
    if call_count > MAX_TOOL_CALLS:
        print(f"第 {call_count} 次调用：超出最大次数 {MAX_TOOL_CALLS}，强制停止")
        break
    print(f"第 {call_count} 次调用：get_weather(city='北京')")

print()
print("工程对策：在调用循环外面包一层计数器，超过阈值就强制终止，返回兜底回复。")
print("另外可以在训练数据里加入'调一次就回复'的正样本，减少模型重复调用的倾向。")

=== 失败模式 4：工具调用死循环 ===
场景：模型连续多次调用 get_weather，每次参数都一样，从不输出最终回复

第 1 次调用：get_weather(city='北京')
第 2 次调用：get_weather(city='北京')
第 3 次调用：get_weather(city='北京')
第 4 次调用：超出最大次数 3，强制停止

工程对策：在调用循环外面包一层计数器，超过阈值就强制终止，返回兜底回复。
另外可以在训练数据里加入'调一次就回复'的正样本，减少模型重复调用的倾向。


## 6. 现代实践：从 Function Call 到标准化工具调用

Function Call 这个名字最早是 OpenAI 在 2023 年 6 月引入的，那时的 API 参数还叫 `functions`。半年后 OpenAI 把它重命名为 Tools，参数变成 `tools`，原本的 `function_call` 字段也改成了 `tool_calls`。这次改名背后是一个更通用的设计：工具不一定是函数，也可以是检索器、代码解释器、文件系统等任何能"被调用"的东西。所以现在社区里更常见的说法是 Tool Use 或 Tool Calling，Function Call 逐渐成了历史叫法。

但各家厂商的 Tool Use 接口仍然各搞各的——OpenAI 一套格式、Anthropic 一套、Google 一套。一个工具要同时支持多个平台，开发者得反复适配。Anthropic 在 2024 年底发布了 MCP（Model Context Protocol），试图统一这个生态。MCP 的核心想法是：把工具的描述和调用协议标准化，让任何 MCP-compatible 的工具都能被任何支持 MCP 的模型直接调用，不再需要为每个模型单独适配。

更进一步的演化是 Agent。当模型能稳定调用工具、外部程序能把结果回填、模型能基于结果决定下一步——这三件事组合起来，就形成了一个"模型自主决策"的循环。这种循环被称为 agent loop。从单纯的 Function Call 到 agent，本质上是同一个机制的递归应用：调用 → 结果 → 决策 → 再调用。

## 小结

这一节所学的内容：

- LLM 有两个根本限制：知识截止（无法知道训练后发生的事）和数值计算错误（不擅长精确算术）
- Function Call 把"决策"和"执行"分开：模型生成结构化的 JSON 调用请求，外部程序执行
- 完整流程六步：定义工具 → 注入 system prompt → 模型输出 JSON → 外部执行 → 结果回填 → 模型生成最终回复
- 多工具场景分两种：并行调用（无依赖，可一次生成多个）和串行调用（有依赖，需分多轮）
- 训练 Function Call 模型需要专门的样本：完整的对话流（system + user + assistant tool call + tool result + assistant final reply），Loss mask 只对 assistant 输出部分计算
- 数据来源三种：人工标注、强模型蒸馏、真实日志
- 常见失败模式四种：工具执行失败、模型选错工具、参数不合法、调用死循环
- 命名演化：Function Call → Tool Use → MCP，背后是从"调函数"到"调任意工具"到"标准化工具接口"的扩展

## 作业

三道题覆盖三类内容：

1. **核心机制**：手写一个 tool call JSON，理解模型输出的结构
2. **训练数据**：标注一个训练样本的 loss mask，理解哪些 token 算 loss
3. **多工具调用**：判断一个场景需要并行还是串行，理解依赖关系

> 可以让 AI 帮忙解释思路，但不建议直接让 AI 完成这道题。

### 作业 1：手写一个 tool call JSON

给定用户问题和工具列表，写出模型应该输出的调用 JSON。

工具列表：

- `calculate(expression)`：执行数学计算
- `search(query)`：网络搜索

用户问题：`123 + 456 × 7 等于多少？`

**小提示**：模型需要决定调用哪个工具、传什么参数。expression 应该是一个数学表达式字符串。

In [15]:
# 作业 1：手写一个 tool call JSON
import json

# TODO：把下面的占位字符串换成你写的 JSON 字符串
# 格式：{"tool_call": {"name": "...", "arguments": {...}}}
your_answer = """{"tool_call": {"name": "calculate", "arguments": {"expression": "(123 + 456) * 7"}}}"""

# 自动校验
assert your_answer != "在这里写你的 JSON 字符串", "请先填入你的答案"

try:
    parsed = json.loads(your_answer)
except json.JSONDecodeError as e:
    raise AssertionError(f"JSON 格式错误：{e}")

assert "tool_call" in parsed, "JSON 顶层应该有 tool_call 字段"
assert parsed["tool_call"]["name"] == "calculate", "应该调用 calculate 工具"
assert "expression" in parsed["tool_call"]["arguments"], "arguments 里应该有 expression 字段"
# 检查表达式是否包含必要部分（不严格要求运算顺序）
expr = parsed["tool_call"]["arguments"]["expression"]
for token in ["123", "456", "7"]:
    assert token in expr, f"表达式里应该包含 {token}"

print("作业 1 通过：你理解了 tool call 的 JSON 结构")
print(f"你的答案：{parsed}")

作业 1 通过：你理解了 tool call 的 JSON 结构
你的答案：{'tool_call': {'name': 'calculate', 'arguments': {'expression': '(123 + 456) * 7'}}}


### 作业 2：训练样本的 loss mask

下面是一个训练样本的对话流（已简化）。对每条消息，判断它对应的 token 在训练时是否应该算 loss。

```
[1] system:           "你可以使用 calculate(expression) 工具"
[2] user:             "12 × 8 等于多少？"
[3] assistant(工具调用): {"name": "calculate", "arguments": {"expression": "12*8"}}
[4] tool:             {"result": 96}
[5] assistant:        "12 × 8 等于 96。"
```

把每条消息对应的 mask 写成一个列表（True 算 loss，False 不算）。

**小提示**：只有模型需要"生成"的位置才该算 loss。输入位置（system/user/tool 结果）不算。

In [16]:
# 作业 2：训练样本的 loss mask

# TODO：把下面的占位列表换成你的答案
# 顺序对应 [1] system, [2] user, [3] assistant(工具调用), [4] tool, [5] assistant
# 每个元素是 True（算 loss）或 False（不算）
your_mask = [False, False, True, False, True]

# 自动校验
assert your_mask != [None, None, None, None, None], "请先填入你的答案"
assert len(your_mask) == 5, "应该有 5 个元素"
assert all(isinstance(x, bool) for x in your_mask), "每个元素应该是 True 或 False"

# 正确答案：system/user/tool 是输入（False），assistant 是输出（True）
expected = [False, False, True, False, True]
assert your_mask == expected, "答案不正确。提示：只有 assistant 的输出位置算 loss"

print("作业 2 通过：你理解了 Function Call 训练的 loss mask")
print(f"你的答案：{your_mask}")
print("解读：[1]system [2]user 是输入；[3]assistant 的工具调用是模型生成的内容；")
print("      [4]tool 是外部程序返回的结果；[5]assistant 的最终回复也是模型生成的。")

作业 2 通过：你理解了 Function Call 训练的 loss mask
你的答案：[False, False, True, False, True]
解读：[1]system [2]user 是输入；[3]assistant 的工具调用是模型生成的内容；
      [4]tool 是外部程序返回的结果；[5]assistant 的最终回复也是模型生成的。


### 作业 3：判断并行还是串行

下面三个场景，分别应该用并行调用还是串行调用？

场景 A：用户问「北京和上海今天分别多少度？」

场景 B：用户问「先查北京今天的天气，然后根据温度推荐穿搭」

场景 C：用户问「123 + 456 等于多少？789 - 123 等于多少？」

把答案写成一个列表，每个元素是字符串 `"并行"` 或 `"串行"`，顺序对应 A/B/C。

**小提示**：判断标准是——后面的工具调用是否依赖前面工具的结果。依赖 → 串行；不依赖 → 并行。

In [17]:
# 作业 3：判断并行还是串行

# TODO：把下面的占位列表换成你的答案
# 顺序对应场景 A/B/C，每个元素是 "并行" 或 "串行"
your_answer = ["并行", "串行", "并行"]

# 自动校验
assert your_answer != [None, None, None], "请先填入你的答案"
assert len(your_answer) == 3, "应该有 3 个元素"
assert all(x in ["并行", "串行"] for x in your_answer), "每个元素应该是 '并行' 或 '串行'"

# 正确答案：
# A：北京和上海的天气互相独立 → 并行
# B：穿搭依赖天气结果 → 串行
# C：两个独立计算 → 并行
expected = ["并行", "串行", "并行"]

# 逐个对比并给出反馈
for i, (y, e, scene) in enumerate(zip(your_answer, expected, ["A", "B", "C"])):
    if y != e:
        hint = {
            "A": "提示：北京和上海的天气互相独立，不依赖",
            "B": "提示：穿搭建议依赖天气结果",
            "C": "提示：两个独立的算术，互不依赖"
        }[scene]
        raise AssertionError(f"场景 {scene} 答错。{hint}")

print("作业 3 通过：你理解了并行和串行的判断标准")
print(f"你的答案：{your_answer}")
print("解读：")
print("  A：两个独立查询，可以同时调用 → 并行")
print("  B：第二个调用依赖第一个的结果 → 串行")
print("  C：两个独立计算 → 并行")

作业 3 通过：你理解了并行和串行的判断标准
你的答案：['并行', '串行', '并行']
解读：
  A：两个独立查询，可以同时调用 → 并行
  B：第二个调用依赖第一个的结果 → 串行
  C：两个独立计算 → 并行
